## Tarea 2: API REST — Base de datos de videojuegos RAWG

ISTALACION DE PACKAGES

In [ ]:
# Packages
import pandas as pd
import os
import urllib.request, json, csv
import numpy as np
from tqdm import tqdm_notebook as tqdm
# For sending GET requests from the API
import requests
# For saving access tokens and for file management when creating and adding to the dataset
import os
# For dealing with json responses we receive from the API
import json
# For displaying the data after
import pandas as pd
# For saving the response data in CSV format
import csv
# For parsing the dates received from twitter in readable formats
import datetime
import dateutil.parser
import unicodedata
#To add wait time between requests
import time
import requests

#  Parte A — Exploración general     (2 puntos)

In [4]:
import requests

url = "https://api.rawg.io/api/games"
params = {
    "key": "1d74b896524c4eb2954455f845f815d1",
    "page_size": 1   # Se usa "1" segun la ducumentacion                                    
}

response = requests.get(url, params=params, timeout=20)
response.raise_for_status()

data = response.json()
print("Total de juegos registrados:", data["count"])

Total de juegos registrados: 898331


#  Parte B — Análisis de categoría     (2 puntos)


B1 — (1 punto) 

Se ordenan los resultados por `-metacritic` para obtener los 5 juegos con mayor puntuación de Metacritic.
Se mostrará el nombre, la valoración (`rating`) y la puntuación de Metacritic (`metacritic`).

In [6]:
import requests
import pandas as pd

url = "https://api.rawg.io/api/games"

# Parámetros para B1
params_b1 = {
    'key': '1d74b896524c4eb2954455f845f815d1',
    'page_size': 5,
    'ordering': '-metacritic'
}

response_b1 = requests.get(url, params=params_b1, timeout=20)
data_b1 = response_b1.json()

# Crear DataFrame y mostrar solo las columnas solicitadas
df_b1 = pd.DataFrame(data_b1['results'])[['name', 'rating', 'metacritic']]

print("--- Top 5 Highest Rated Games (Metacritic) ---")
print(df_b1)

--- Top 5 Highest Rated Games (Metacritic) ---
                                   name  rating  metacritic
0  The Legend of Zelda: Ocarina of Time    4.38          99
1                    Soulcalibur (1998)    0.00          98
2                           Soulcalibur    4.38          98
3                     Baldur's Gate III    4.44          97
4                         Metroid Prime    4.35          97


In [9]:
print (response_b1.status_code) 

200


B2 - (1 punto)

Aquí añadimos un filtro extra. Según la documentación de RAWG, el parámetro para filtrar por tienda es stores. Como nos piden específicamente Steam, usamos el ID 1.

In [10]:
import requests
import pandas as pd

url = "https://api.rawg.io/api/games"


# Parámetros para B2
params_b2 = {
    'key': '1d74b896524c4eb2954455f845f815d1',
    'page_size': 10,
    'stores': 1,           # ID de Steam
    'ordering': '-rating'  # Los "mejores" suele referirse al rating de usuarios
}

response_b2 = requests.get(url, params=params_b2, timeout=20)
data_b2 = response_b2.json()

# Crear DataFrame y mostrar solo las columnas solicitadas
df_b2 = pd.DataFrame(data_b2['results'])[['name', 'rating', 'metacritic']]

print("\n--- 10 Best Games on Steam ---")
print(df_b2)


--- 10 Best Games on Steam ---
                                                name  rating  metacritic
0  Warhammer 40,000: Dawn of War - Definitive Edi...    4.83         NaN
1                     No Case Should Remain Unsolved    4.83         NaN
2          The Witcher 3: Wild Hunt – Blood and Wine    4.81        92.0
3         The Witcher 3 Wild Hunt - Complete Edition    4.80        92.0
4         The Witcher 3: Wild Hunt – Hearts of Stone    4.76        90.0
5                                    Persona 5 Royal    4.75        94.0
6                                      Guilty Parade    4.71         NaN
7                    Cyberpunk 2077: Phantom Liberty    4.71         NaN
8                   The Binding of Isaac: Repentance    4.69         NaN
9                                       Red Matter 2    4.67         NaN


In [11]:
print(response_b2.status_code)

200


# Parte C — Comparaciones     (3 pts)

C1 — (0,5 puntos)

Para este punto, haremos dos peticiones separadas y compararemos el promedio de su "rating".

In [19]:
import requests
url = "https://api.rawg.io/api/games"

def get_top_5_platform(plat_id):
    params_c1 = {'key': '1d74b896524c4eb2954455f845f815d1', 'page_size': 5, 'platforms': plat_id, 'ordering': '-rating'}
    response_c1 = requests.get(url, params=params_c1, timeout=20)
    data_c1 = response_c1.json()
    return pd.DataFrame(data_c1['results'])

df_pc = get_top_5_platform(4)
df_ps5 = get_top_5_platform(187)

avg_pc = df_pc['rating'].mean()
avg_ps5 = df_ps5['rating'].mean()

print(f"Rating promedio PC: {avg_pc:.2f}")
print(f"Rating promedio PS5: {avg_ps5:.2f}")
print(f"La plataforma con mejor valoración es: {'PC' if avg_pc > avg_ps5 else 'PS5'}")

Rating promedio PC: 4.83
Rating promedio PS5: 4.72
La plataforma con mejor valoración es: PC


C2 — (0,5 puntos)

Elegiremos 3 títulos conocidos usando el parámetro "search".

In [25]:
import requests
url = "https://api.rawg.io/api/games"

juegos_famosos = ["The Witcher 3", "Elden Ring", "Grand Theft Auto V"]
lista_comparativa = []

for juego in juegos_famosos:
    params_c2    = {'key': '1d74b896524c4eb2954455f845f815d1', 'search': juego, 'page_size': 1}
    response_c2 = requests.get(url, params=params_c2, timeout=20)
    data_c2 = response_c2.json()
    res_c2 = data_c2['results'][0]

    # Extraer nombres de géneros y plataformas (vienen como listas de dicts)
    generos = ", ".join([g['name'] for g in res_c2['genres']])
    plataformas = ", ".join([p['platform']['name'] for p in res_c2['platforms']])
    
    lista_comparativa.append({
        'nombre': res_c2['name'],
        'calificacion': res_c2['rating'],
        'Metacritic': res_c2 ['metacritic'],
        'géneros': generos,
        'plataformas': plataformas
    })

df_comparativo = pd.DataFrame(lista_comparativa)
display(df_comparativo)

,nombre,calificacion,Metacritic,géneros,plataformas
0,The Witcher 3: Wild Hunt,4.64,92,"Action, RPG","PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."
1,Elden Ring,4.38,95,"Action, RPG","PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."
2,Grand Theft Auto V,4.47,92,Action,"PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."


C3 — (0,5 puntos)

Calcularemos cuál género tiene el mejor promedio basado en su Top 5. 

In [30]:
import requests
url = "https://api.rawg.io/api/games"
api_key = '1d74b896524c4eb2954455f845f815d1'

generos_ids_c3 = {'Acción': 4, 'Indie': 51, 'RPG': 5, 'Shooter': 2}
resultados_gen_c3= {}

for nombre, g_id in generos_ids_c3.items():
    params_c3 = {'key': api_key, 'page_size': 5, 'genres': g_id, 'ordering': '-rating'}
    res_c3 = requests.get(url, params=params_c3, timeout=20).json()['results']
    promedio_c3 = pd.DataFrame(res_c3)['rating'].mean()
    resultados_gen_c3[nombre] = promedio_c3    

print("Promedios por género:")
for genero, promedio_c3 in resultados_gen_c3.items():
    print(f"- {genero}: {float(promedio_c3):.2f}")
    
print(f"\nEl género con mejores juegos según usuarios es: {mejor_genero_c3}")

Promedios por género:
- Acción: 4.77
- Indie: 4.72
- RPG: 4.80
- Shooter: 4.68

El género con mejores juegos según usuarios es: RPG


C4 — (0,5 puntos)

Usaremos el parámetro dates

In [33]:
import requests
url = "https://api.rawg.io/api/games"
api_key = '1d74b896524c4eb2954455f845f815d1'

años_c4 = [2021, 2022, 2023]
mejor_año_meta_c4 = {}

for año in años_c4:
    params_c4 = {'key': api_key, 'dates': f'{año}-01-01,{año}-12-31', 'ordering': '-metacritic', 'page_size': 10}
    res_c4 = requests.get(url  , params=params_c4, timeout=20).json()['results']
    promedio_meta_c4 = pd.DataFrame(res_c4)['metacritic'].mean()
    mejor_año_meta_c4[año] = promedio_meta_c4


# identificar el mejor año
año_mejor_valorado = max(mejor_año_meta_c4, key=mejor_año_meta_c4.get)

# impresión más presentable
print("=== Puntuación Metacritic media por año ===")
for año, promedio in mejor_año_meta_c4.items():
    print(f"{año}: {float(promedio):.2f}")

print("\n=== Comparación final ===")
print(
    f"El año con los juegos mejor valorados fue {año_mejor_valorado}, "
    f"con una puntuación media de Metacritic de {float(mejor_año_meta_c4[año_mejor_valorado]):.2f}."
)

=== Puntuación Metacritic media por año ===
2021: 88.20
2022: 88.30
2023: 89.50

=== Comparación final ===
El año con los juegos mejor valorados fue 2023, con una puntuación media de Metacritic de 89.50.


C5 — (1.0 pt)

In [35]:
import requests
import pandas as pd
import os   
url = "https://api.rawg.io/api/games"
api_key = '1d74b896524c4eb2954455f845f815d1'

# 1. Obtener los 20 mejores de todos los tiempos
params_final_c5 = {'key': api_key, 'page_size': 20, 'ordering': '-rating'}
data_final_c5 = requests.get(url, params=params_final_c5).json()['results']

# 2. Procesar para incluir solo las columnas pedidas

final_list_c5 = []
for g in data_final_c5:
    final_list_c5.append({
        'name': g['name'],
        'rating': g['rating'],
        'metacritic': g.get('metacritic'),
        'release_date': g['released'],
        'main_genre': g['genres'][0]['name'] if g['genres'] else "N/A"
    })

df_final_c5 = pd.DataFrame(final_list_c5)

# 3. Guardar en la ruta específica
os.makedirs("output", exist_ok=True)
df_final_c5.to_csv("output/top20_rawg.csv", index=False)

# 4. Mostrar las primeras 5 filas
print("--- Primeras 5 filas del archivo generado ---")
display(df_final_c5.head(5))

--- Primeras 5 filas del archivo generado ---


,name,rating,metacritic,release_date,main_genre
0,The Elder Scrolls VI,4.86,NaN,None,Action
1,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN,None,N/A
2,No Case Should Remain Unsolved,4.83,NaN,2024-01-17,Adventure
3,Gimmick!,4.83,NaN,1992-01-31,N/A
4,Super Robot Taisen: Original Generation,4.83,NaN,2002-11-22,Action
